# 03 — Semantic search: encode movies and run the mood queries (Google Colab)

Runs `src/semantic.py` + `src/run_semantic_eval.py` with a **GPU**.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Put the repository folder on Google Drive, e.g. `MyDrive/Movie-recommendation-system/`
   (GitHub → Code → Download ZIP → unzip → upload the folder). For the full run, also put
   `clean_movies.parquet` into its `data/processed/` folder.
3. Run the cells top to bottom. **Part A (sample)** takes ~1–2 min. **Part B (full data)** is optional and long.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/Movie-recommendation-system'   # <- change if your folder is elsewhere
%cd {REPO}
!ls

In [ ]:
!pip -q install sentence-transformers pyarrow
import torch, platform
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Part A — 100-movie sample (do this first)

Encodes metadata+plot and subtitle chunks (150 words) with `all-MiniLM-L6-v2`.

In [ ]:
!python -m src.semantic train --data data/processed/sample_clean_movies.csv --output models/semantic_sample --overwrite

In [ ]:
# A few quick sanity queries, including TF-IDF's known failures (misspelling, negation, vague query)
for q in ["gloooomy thriling mistree", "I want something dark but not scary", "I feel like watching something."]:
    print("=" * 100, "\nQUERY:", q)
    !python -m src.semantic search --index models/semantic_sample --query "{q}" --source combined --top-k 3

In [ ]:
# All 28 queries x 4 variants -> evaluation/results/semantic1/sem_*_mood_rankings.csv + new pairs in mood_judgments.csv
!python -m src.run_semantic_eval run --index models/semantic_sample

**Now download / sync these files back to the repo and send them to Tomáš:**
`evaluation/results/semantic1/sem_*_mood_rankings.csv`, `semantic_query_status.csv`, `evaluation/results/comparison/method_overlap.csv`,
`evaluation/mood_judgments.csv`, `models/semantic_sample/manifest.json`.

Do **not** commit `chunk_text.parquet` or the `.npy` files of the full index (size + subtitle copyright).

## Part B — full dataset (optional, long)

First a **timing test on 500 movies** to estimate the full run. Full run ≈ (time for 500) × (N / 500).
The index is written to Drive only at the end. If Colab disconnects during encoding (free tier limit),
rerun with `--max-chunks-per-movie 20` (20 evenly spaced chunks per movie, ~2× faster) and record
this trade-off in the failure journal.

In [ ]:
import time, pyarrow.parquet as pq
FULL = 'data/processed/clean_movies.parquet'
print('movies in full data:', pq.ParquetFile(FULL).metadata.num_rows)
t = time.time()
!python -m src.semantic train --data {FULL} --max-rows 500 --output models/semantic_timing --overwrite --batch-size 128
print(f'500 movies: {time.time()-t:.0f}s')

In [ ]:
# Full index (the long step). Does not touch any results or judgments.
!python -m src.semantic train --data {FULL} --output models/semantic_full --overwrite --batch-size 128

**Only after the team agrees:** run the queries on the full index. Outputs go to SEPARATE files
(`evaluation/results/full/`, `evaluation/mood_judgments_full.csv`), so the sample results and the
labels being filled in are never overwritten.

In [ ]:
# !python -m src.run_semantic_eval run --index models/semantic_full --reference {FULL} --results-dir evaluation/results/full --judgments evaluation/mood_judgments_full.csv